# PHASE 4 — UNSUPERVISED LEARNING


# Day 23 — Hierarchical Clustering


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Explain the concept of Agglomerative (Bottom-Up) Clustering.
- Build and interpret a **Dendrogram** to visualize the grouping of data.
- Understand how to "cut" the dendrogram tree to get your final clusters.
- Contrast Hierarchical Clustering with K-Means and DBSCAN.


## 2. Prerequisites
- Day 21 & 22 (K-Means, DBSCAN, Silhouette Score).


## 3. Concept: Hierarchical Clustering
Yesterday we looked at DBSCAN (density). The day before, K-Means (Centroids). 
What if we don't want to guess the number of clusters upfront, and we want to see a visual map of how every single point relates to every other point?

**Agglomerative (Hierarchical) Clustering** works "bottom-up":
1. It starts by assuming every single data point is its own tiny cluster (If you have 100 points, you have 100 clusters).
2. It finds the 2 closest clusters and merges them together (Now you have 99 clusters).
3. It repeats this process over and over, merging the closest clusters until there is only 1 massive cluster left that contains everything.

It records every single merge along the way!


## 4. Concept: The Dendrogram
Because it records every merge, we can plot the entire history as a massive family tree called a **Dendrogram**. 
The height of the branches on the tree represents the physical distance between the clusters that were merged. 

By looking at the tree, you can visually spot where the major splits occur, and simply draw a horizontal line across the tree to "cut" it into whatever number of clusters makes visual sense!


## 5. Scikit-learn API
```python
from sklearn.cluster import AgglomerativeClustering
model = AgglomerativeClustering(n_clusters=3, linkage='ward')
```
*Note: Scikit-learn doesn't draw Dendrograms. We use the `scipy` library to draw the tree.*


## 6. Simple Example: The Dendrogram
Let's generate some blob data, scale it, and then use `scipy` to draw the entire hierarchical tree before we even build our Scikit-learn model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

# 1. Generate 3 distinct blobs of data
X, y = make_blobs(n_samples=50, centers=3, cluster_std=0.8, random_state=42)

# 2. Scale the data (Mandatory for distance-based clustering!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Calculate the linkage (the merging history)
# 'ward' minimizes the variance of the clusters being merged
Z = linkage(X_scaled, method='ward')

# 4. Plot the Dendrogram
plt.figure(figsize=(10, 5))
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Data Point Index (or cluster size)')
plt.ylabel('Distance (Height)')
dendrogram(Z)
plt.axhline(y=5, color='r', linestyle='--', label='Cut Line (3 Clusters)')
plt.legend()
plt.show()


## 7. Code Walkthrough
- We only used 50 samples so the tree is readable.
- `linkage(X_scaled, 'ward')` did all the math to merge the points step-by-step.
- `dendrogram(Z)` drew the tree.
- Notice how the tree naturally splits into exactly 3 major branches (colored orange and green). The vertical lines are very tall, meaning there is a large distance between these 3 major groups. This perfectly matches the `centers=3` we asked for!


## 8. Experiment: Training the Model
Now that we visually see 3 is the correct number of clusters, let's use Scikit-learn to actually assign the labels to the data.


In [ ]:
from sklearn.cluster import AgglomerativeClustering

# 1. Train the Model
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels = agg.fit_predict(X_scaled)

# 2. Plot the resulting clusters
plt.figure(figsize=(6, 4))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='viridis', s=50)
plt.title('Agglomerative Clustering (K=3)')
plt.show()


> It perfectly identified the 3 groups. Unlike K-Means, it didn't need to randomly drop Centroids and move them around. It just followed the tree structure down from the top!


## 9. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
agg_2 = AgglomerativeClustering(n_clusters=2, linkage='ward')
labels_2 = agg_2.fit_predict(X_scaled)


> **Question:** Look back at the Dendrogram in Section 6. If we tell the model to find `n_clusters=2`, what will the tree do?

**Think before running the next cell!**


In [ ]:
print('If n_clusters=2, the algorithm simply moves the horizontal cut line higher up the tree.')
print('It will merge the two closest major branches (likely the two on the right side of the dendrogram) into a single massive cluster, leaving the left branch as the second cluster.')


## 10. Coding Exercise
Prove the concept above. Plot the scatter plot of `X_scaled`, colored by `labels_2`. You will see two of the blobs merged into one.


In [ ]:
# YOUR CODE HERE
plt.figure(figsize=(6, 4))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_2, cmap='viridis', s=50)
plt.title('Agglomerative Clustering (K=2)')
plt.show()
print('As predicted, the horizontal cut line moved up, merging two distinct blobs into a single cluster.')


## 11. Debugging Challenge
A scientist is trying to use Agglomerative Clustering on an astronomical dataset of 5,000,000 stars to find constellations. The code crashes with an `Out of Memory (OOM)` error. Why?


In [ ]:
# Conceptual Bug
print('Error: Agglomerative Clustering requires a massive distance matrix.')
print('To merge the closest clusters, it has to know the exact distance between EVERY single point and EVERY other point.')
print('For 5,000,000 stars, the distance matrix requires (5M * 5M) / 2 calculations. That takes terabytes of RAM!')


> **Rule:** Hierarchical Clustering is brilliant for visualization, but it scales terribly. Do not use it on datasets larger than a few thousand rows. Stick to K-Means for massive datasets.


## 12. Clustering Algorithm Comparison
You now know three clustering algorithms:
1. **K-Means**: Fast, scalable, forces everything into a circle. Good default.
2. **DBSCAN**: Finds weird shapes, automatically tags outliers (`-1`). Hard to tune `eps`. Fails if densities vary.
3. **Hierarchical**: Gives a beautiful visual tree (Dendrogram) to help you pick K. Horrible for large datasets.


## 13. Real-World Example
**Biology (Genetics)**: Hierarchical clustering is the absolute standard in biology for DNA analysis. If you have the genetic sequences of 100 different species, you run Agglomerative Clustering. The resulting Dendrogram is literally the Evolutionary Tree of Life! It shows exactly which species mutated from which ancestors based on genetic distance.


## 14. Mini Project
Build a Pipeline with `StandardScaler` and `AgglomerativeClustering(n_clusters=4)`. Test it on the data below and print the final Silhouette Score.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

X_proj, _ = make_blobs(n_samples=500, centers=4, cluster_std=0.5, random_state=42)

agg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('agg', AgglomerativeClustering(n_clusters=4, linkage='ward'))
])

# Note: Unsupervised models don't have a .predict() for new data (unless it's K-means).
# We must use .fit_predict() on the exact data we want to label.
labels_proj = agg_pipe.fit_predict(X_proj)

score = silhouette_score(X_proj, labels_proj)
print(f'Hierarchical Clustering Silhouette Score: {score:.3f}')


## 15. Common Mistakes
- **Not Scaling Data**: Like all distance-based models, `StandardScaler` is required.
- **Calling `.predict()`**: Agglomerative Clustering cannot predict new, unseen data points. It doesn't learn an equation or save a Centroid. It only groups the data it is currently looking at. You must use `.fit_predict()`.
- **Using on Big Data**: Crashing the server because the distance matrix exceeds RAM.


## 16. Interview Questions
- **Beginner**: How does a Dendrogram help you choose the number of clusters? (Answer: You look for the longest vertical lines and cut horizontally through them).
- **Intermediate**: Why can K-Means predict on new data, but Agglomerative Clustering cannot? (Answer: K-Means saves the physical coordinates of its Centroids. Agglomerative only remembers the merging history of the specific data it trained on).
- **Advanced**: Explain the 'ward' linkage method. (Answer: Instead of merging the two clusters that are physically closest, 'ward' merges the two clusters that will result in the smallest increase in overall variance).


## 17. Knowledge Check
- What is the tree diagram used in Hierarchical Clustering called? (Dendrogram)
- Does Agglomerative clustering scale well to 1 million rows? (No, it causes memory errors)


## 18. Summary
- **Agglomerative Clustering** is a bottom-up approach that merges points step-by-step.
- It builds a **Dendrogram** showing the entire history of the dataset.
- It is fantastic for visualization and choosing $K$.
- It does not have a `.predict()` method for new data.
- It is extremely computationally expensive on large datasets.


## 19. Homework
Load the `load_wine` dataset. Scale it. Run `AgglomerativeClustering(n_clusters=3)` and get the labels. Calculate the `silhouette_score` and compare it to the score you got when you ran K-Means on the wine dataset in Day 21!
